<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [6]:
using System;
using System.Collections.Generic;
using System.Linq;
public delegate void MovieActionHandler<T>(T movie, string action) where T : Movie;
public delegate bool MovieFilter<T>(T movie) where T : Movie;
public delegate void MovieCollectionChangedHandler(string collectionName, string changeType, int newCount);

public interface IMovieRepository<T> where T : Movie
{
    void AddMovie(T movie);
    void RemoveMovie(string title);
    T FindMovie(string title);
    List<T> GetMoviesByRating(double minRating);
    List<T> FilterMovies(MovieFilter<T> filter);
    void DisplayAllMovies();
    int Count { get; }
    event Action<T> MovieAdded;
    event Action<T> MovieRemoved;
    event Action CollectionChanged;
}

public interface IMovieResourceManager : IDisposable
{
    void LoadResources();
    void ReleaseResources();
}

public class MovieCollection<T> : IMovieRepository<T>, IMovieResourceManager where T : Movie
{
    private List<T> _movies;
    private bool _disposed = false;
    
    public event Action<T> MovieAdded;
    public event Action<T> MovieRemoved;
    public event Action CollectionChanged;
    public static event MovieCollectionChangedHandler GlobalCollectionChanged;

    public MovieCollection()
    {
        _movies = new List<T>();
        LoadResources();
    }

    public void AddMovie(T movie)
    {
        _movies.Add(movie);
        Console.WriteLine($"Фильм '{movie.Title}' добавлен в коллекцию");
        
        movie.MovieViewed += OnMovieViewed;
        
        MovieAdded?.Invoke(movie);
        CollectionChanged?.Invoke();
        GlobalCollectionChanged?.Invoke(GetType().Name, "Added", _movies.Count);
    }

    public void RemoveMovie(string title)
    {
        var movie = _movies.FirstOrDefault(m => m.Title.Equals(title, StringComparison.OrdinalIgnoreCase));
        if (movie != null)
        {
            _movies.Remove(movie);
            Console.WriteLine($"Фильм '{title}' удален из коллекции");
            
            movie.MovieViewed -= OnMovieViewed;
            
            MovieRemoved?.Invoke(movie);
            CollectionChanged?.Invoke();
            GlobalCollectionChanged?.Invoke(GetType().Name, "Removed", _movies.Count);
        }
    }

    public T FindMovie(string title)
    {
        return _movies.FirstOrDefault(m => m.Title.Equals(title, StringComparison.OrdinalIgnoreCase));
    }

    public List<T> GetMoviesByRating(double minRating)
    {
        return _movies.Where(m => m.Rating >= minRating).ToList();
    }

    public List<T> FilterMovies(MovieFilter<T> filter)
    {
        return _movies.Where(movie => filter(movie)).ToList();
    }

    public void DisplayAllMovies()
    {
        Console.WriteLine($"\n=== КОЛЛЕКЦИЯ ФИЛЬМОВ ({_movies.Count} шт.) ===");
        foreach (var movie in _movies)
        {
            Console.WriteLine($"- {movie.Title} ({movie.ReleaseYear}) - {movie.Rating:F1}/10 - Просмотров: {movie.ViewCount}");
        }
    }

    public int Count => _movies.Count;

    private void OnMovieViewed(Movie movie)
    {
        Console.WriteLine($" Уведомление: фильм '{movie.Title}' просмотрен! Всего просмотров: {movie.ViewCount}");
    }

    void IDisposable.Dispose()
    {
        Dispose(true);
        GC.SuppressFinalize(this);
    }

    protected virtual void Dispose(bool disposing)
    {
        if (!_disposed)
        {
            if (disposing)
            {
                ReleaseResources();
                _movies.Clear();
                Console.WriteLine("Ресурсы MovieCollection освобождены");
            }
            _disposed = true;
        }
    }

    public void LoadResources()
    {
        Console.WriteLine("Ресурсы коллекции загружены");
    }

    public void ReleaseResources()
    {
        Console.WriteLine("Ресурсы коллекции освобождены");
    }

    ~MovieCollection()
    {
        Dispose(false);
    }
}

public class MovieService<T> where T : Movie
{
    private readonly IMovieRepository<T> _repository;

    public MovieService(IMovieRepository<T> repository)
    {
        _repository = repository ?? throw new ArgumentNullException(nameof(repository));
    }

    public void AddMovieWithValidation(T movie)
    {
        if (movie.Rating < 0 || movie.Rating > 10)
            throw new ArgumentException("Рейтинг должен быть от 0 до 10");

        if (string.IsNullOrWhiteSpace(movie.Title))
            throw new ArgumentException("Название фильма не может быть пустым");

        _repository.AddMovie(movie);
    }

    public void DisplayHighRatedMovies(double minRating)
    {
        var movies = _repository.GetMoviesByRating(minRating);
        Console.WriteLine($"\n=== ФИЛЬМЫ С РЕЙТИНГОМ ВЫШЕ {minRating:F1} ===");
        foreach (var movie in movies)
        {
            Console.WriteLine($"- {movie.Title} ({movie.Rating:F1}/10)");
        }
    }

    public T FindMovieByTitle(string title)
    {
        return _repository.FindMovie(title);
    }
}

public class AdvancedMovieService<T> where T : Movie
{
    private readonly IMovieRepository<T> _repository;
    
    public MovieActionHandler<T> OnCustomAction { get; set; }

    public AdvancedMovieService(IMovieRepository<T> repository)
    {
        _repository = repository ?? throw new ArgumentNullException(nameof(repository));
        
        if (repository is MovieCollection<T> collection)
        {
            collection.MovieAdded += OnMovieAddedToCollection;
            collection.MovieRemoved += OnMovieRemovedFromCollection;
            MovieCollection<T>.GlobalCollectionChanged += OnGlobalCollectionChanged;
        }
    }

    public void AddMovieWithValidation(T movie)
    {
        if (movie.Rating < 0 || movie.Rating > 10)
            throw new ArgumentException("Рейтинг должен быть от 0 до 10");

        if (string.IsNullOrWhiteSpace(movie.Title))
            throw new ArgumentException("Название фильма не может быть пустым");

        _repository.AddMovie(movie);
    }

    public void PerformCustomAction(T movie, string action)
    {
        OnCustomAction?.Invoke(movie, action);
    }

    public List<T> FindMoviesByTag(string tag)
    {
        return _repository.FilterMovies(movie => movie.Tags.Any(t => t.Contains(tag, StringComparison.OrdinalIgnoreCase)));
    }

    public List<T> FindPopularMovies(int minViews = 100)
    {
        return _repository.FilterMovies(movie => movie.ViewCount >= minViews);
    }

    public List<T> FindRecentMovies(int days = 30)
    {
        var cutoffDate = DateTime.Now.AddDays(-days);
        return _repository.FilterMovies(movie => movie.AddedDate >= cutoffDate);
    }

    public void SimulateMovieViews(T movie, int viewCount)
    {
        for (int i = 0; i < viewCount; i++)
        {
            movie.IncrementViews();
        }
        Console.WriteLine($"Симулировано {viewCount} просмотров для '{movie.Title}'");
    }

    private void OnMovieAddedToCollection(T movie)
    {
        Console.WriteLine($" Сервис: Фильм '{movie.Title}' добавлен в коллекцию");
    }

    private void OnMovieRemovedFromCollection(T movie)
    {
        Console.WriteLine($" Сервис: Фильм '{movie.Title}' удален из коллекцию");
    }

    private void OnGlobalCollectionChanged(string collectionName, string changeType, int newCount)
    {
        Console.WriteLine($" Глобальное изменение: {collectionName} - {changeType} (всего: {newCount})");
    }
}

public class MovieCollectionManager : IDisposable
{
    private List<IMovieResourceManager> _managers = new List<IMovieResourceManager>();

    public void RegisterCollection(IMovieResourceManager manager)
    {
        _managers.Add(manager);
    }

    public void Dispose()
    {
        foreach (var manager in _managers)
        {
            manager?.Dispose();
        }
        _managers.Clear();
    }
}

public class Movie
{
    private string _title;
    private string _director;
    private int _releaseYear;
    private double _rating;
    private int _duration;
    private string _country;
    private List<string> _actors;
    private string _producer;
    private string _language;
    private decimal _budget;
    private List<string> _tags;
    private int _viewCount;
    private DateTime _addedDate;

    public Movie(string title, string director, int releaseYear, int duration, string country)
    {
        Title = title;
        Director = director;
        ReleaseYear = releaseYear;
        Duration = duration;
        Country = country;
        _rating = 0;
        _actors = new List<string>();
        _producer = "Неизвестно";
        _language = "Английский";
        _budget = 0;
        _tags = new List<string>();
        _viewCount = 0;
        _addedDate = DateTime.Now;
    }

    public string Title
    {
        get => _title;
        set => _title = !string.IsNullOrWhiteSpace(value) ? value : "Неизвестный фильм";
    }

    public string Director
    {
        get => _director;
        set => _director = !string.IsNullOrWhiteSpace(value) ? value : "Неизвестный режиссер";
    }

    public int ReleaseYear
    {
        get => _releaseYear;
        set => _releaseYear = value >= 1888 ? value : 1888;
    }

    public int Duration
    {
        get => _duration;
        set => _duration = value > 0 ? value : 60;
    }

    public string Country
    {
        get => _country;
        set => _country = !string.IsNullOrWhiteSpace(value) ? value : "Неизвестно";
    }

    public double Rating
    {
        get => _rating;
        protected set => _rating = Math.Clamp(value, 0, 10);
    }

    public string Producer
    {
        get => _producer;
        set => _producer = !string.IsNullOrWhiteSpace(value) ? value : "Неизвестно";
    }

    public string Language
    {
        get => _language;
        set => _language = !string.IsNullOrWhiteSpace(value) ? value : "Английский";
    }

    public decimal Budget
    {
        get => _budget;
        set => _budget = value >= 0 ? value : 0;
    }

    public List<string> Tags => new List<string>(_tags);
    public int ViewCount => _viewCount;
    public DateTime AddedDate => _addedDate;
    public virtual string AgeRating => "PG-13"; 

    public List<string> Actors => new List<string>(_actors);

    public virtual void AddActor(string actor)
    {
        if (!string.IsNullOrWhiteSpace(actor) && !_actors.Contains(actor))
        {
            _actors.Add(actor);
        }
    }

    public virtual void AddActor(string actor, string role)
    {
        if (!string.IsNullOrWhiteSpace(actor))
        {
            _actors.Add($"{actor} ({role})");
            Console.WriteLine($"Актер {actor} добавлен в роль {role}");
        }
    }

    public void AddActors(params string[] actors)
    {
        foreach (var actor in actors)
        {
            AddActor(actor);
        }
    }

    public virtual void AddTag(string tag)
    {
        if (!string.IsNullOrWhiteSpace(tag) && !_tags.Contains(tag))
        {
            _tags.Add(tag);
            Console.WriteLine($"Добавлен тег '{tag}' к фильму '{Title}'");
        }
    }

    public void AddTags(params string[] tags)
    {
        foreach (var tag in tags)
        {
            AddTag(tag);
        }
    }

    public virtual void IncrementViews()
    {
        _viewCount++;
        OnMovieViewed(); 
    }

    public string GetPopularityStatus()
    {
        return ViewCount switch
        {
            < 100 => "Низкая популярность",
            < 1000 => "Средняя популярность",
            < 10000 => "Высокая популярность",
            _ => "Бестселлер"
        };
    }

    public virtual decimal CalculateProductionCost()
    {
        return Budget * 0.7m; 
    }

    public virtual decimal CalculateRevenue()
    {
        return Budget * 2; 
    }

    public virtual decimal CalculateRevenue(decimal multiplier)
    {
        return Budget * multiplier;
    }

    public event Action<Movie> MovieViewed;

    protected virtual void OnMovieViewed()
    {
        MovieViewed?.Invoke(this);
    }

    public string GetDurationInfo()
    {
        int hours = Duration / 60;
        int minutes = Duration % 60;
        return $"{hours}ч {minutes}мин";
    }

    public int GetMovieAge()
    {
        return DateTime.Now.Year - ReleaseYear;
    }

    public bool IsClassic()
    {
        return GetMovieAge() >= 25;
    }

    public virtual string GetInfo()
    {
        return $"Фильм: {Title}\nРежиссер: {Director}\nГод: {ReleaseYear}";
    }

    public virtual string GetInfo(bool includeDetails)
    {
        if (!includeDetails)
            return GetInfo();
        
        return $"{GetInfo()}\nПродолжительность: {GetDurationInfo()}\nСтрана: {Country}\nБюджет: ${Budget:N0}";
    }

    public virtual void Watch()
    {
        Console.WriteLine($"Смотрим: {Title}");
    }

    public virtual void Watch(string viewingFormat)
    {
        Console.WriteLine($"Смотрим: {Title} в формате {viewingFormat}");
    }

    public virtual void Rate(double score)
    {
        Rating = score;
        Console.WriteLine($"Оценка '{Title}': {Rating:F1}/10");
    }

    public virtual void Rate(double score, string comment)
    {
        Rate(score);
        Console.WriteLine($"Комментарий: {comment}");
    }

    public void DisplayCast()
    {
        if (_actors.Count > 0)
        {
            Console.WriteLine($"Актеры ({_actors.Count}): {string.Join(", ", _actors)}");
        }
    }
}

public class Documentary : Movie
{
    private string _theme;
    private bool _isEducational;
    private int _interviewCount;
    private string _researchMethod;
    private bool _hasScientificReview;
    private string _fundingSource;

    public Documentary(string title, string director, int releaseYear, int duration, string country, string theme, bool isEducational)
        : base(title, director, releaseYear, duration, country)
    {
        Theme = theme;
        IsEducational = isEducational;
        InterviewCount = 0;
        ResearchMethod = "Полевые исследования";
        HasScientificReview = false;
        FundingSource = "Неизвестно";
    }

    public string Theme
    {
        get => _theme;
        set => _theme = !string.IsNullOrWhiteSpace(value) ? value : "Общая тематика";
    }

    public bool IsEducational
    {
        get => _isEducational;
        set => _isEducational = value;
    }

    public int InterviewCount
    {
        get => _interviewCount;
        set => _interviewCount = value >= 0 ? value : 0;
    }

    public string ResearchMethod
    {
        get => _researchMethod;
        set => _researchMethod = !string.IsNullOrWhiteSpace(value) ? value : "Не указан";
    }

    public bool HasScientificReview
    {
        get => _hasScientificReview;
        set => _hasScientificReview = value;
    }

    public string FundingSource
    {
        get => _fundingSource;
        set => _fundingSource = !string.IsNullOrWhiteSpace(value) ? value : "Неизвестно";
    }

    public override string AgeRating => "PG"; 

    public void ConductInterview(string interviewee)
    {
        InterviewCount++;
        Console.WriteLine($"Проведено интервью с {interviewee}. Всего интервью: {InterviewCount}");
    }

    public string GetEducationalValue()
    {
        return IsEducational ? "Образовательный фильм" : "Развлекательный документальный фильм";
    }
    public void ApproveScientificReview()
    {
        HasScientificReview = true;
        Console.WriteLine($"Научный обзор для '{Title}' одобрен");
    }

    public string GetFundingInfo()
    {
        return FundingSource.ToLower() switch
        {
            "государственный" => "Финансируется государством",
            "частный" => "Частное финансирование",
            "краудфандинг" => "Народное финансирование",
            _ => "Источник финансирования неизвестен"
        };
    }

    public override decimal CalculateProductionCost()
    {
        decimal baseCost = base.CalculateProductionCost();
        return baseCost * 0.6m;
    }

    public override void IncrementViews()
    {
        base.IncrementViews();
        if (IsEducational)
        {
            Console.WriteLine($"Образовательный контент '{Title}' просмотрен!");
        }
    }

    public override void Watch()
    {
        base.Watch();
        Console.WriteLine($"Тема: {Theme}");
        Console.WriteLine(GetEducationalValue());
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $"\nТип: Документальный\nТема: {Theme}";
    }

    public override string GetInfo(bool includeDetails)
    {
        if (!includeDetails)
            return GetInfo();
        
        return base.GetInfo(includeDetails) + $"\nТема: {Theme}\nОбразовательный: {IsEducational}\nМетод исследования: {ResearchMethod}";
    }

    public override void Rate(double score)
    {
        double documentaryBonus = 0.5;
        base.Rate(score + documentaryBonus);
        Console.WriteLine("(Бонус за документальный жанр: +0.5)");
    }

    public void UpdateResearch(string newMethod, int newInterviews)
    {
        ResearchMethod = newMethod;
        InterviewCount = newInterviews;
        Console.WriteLine($"Метод исследования обновлен: {newMethod}, интервью: {newInterviews}");
    }
}

public class FeatureFilm : Movie
{
    private string _genre;
    private bool _hasSequel;
    private string _screenplayAuthor;
    private List<string> _sequels;
    private List<string> _awards;
    private bool _isBlockbuster;

    public FeatureFilm(string title, string director, int releaseYear, int duration, string country, string genre, decimal budget)
        : base(title, director, releaseYear, duration, country)
    {
        Genre = genre;
        Budget = budget;
        HasSequel = false;
        _sequels = new List<string>();
        _awards = new List<string>();
        _isBlockbuster = budget > 100000000m; 
    }

    public string Genre
    {
        get => _genre;
        set => _genre = !string.IsNullOrWhiteSpace(value) ? value : "Драма";
    }

    public bool HasSequel
    {
        get => _hasSequel;
        set => _hasSequel = value;
    }

    public string ScreenplayAuthor
    {
        get => _screenplayAuthor;
        set => _screenplayAuthor = !string.IsNullOrWhiteSpace(value) ? value : "Неизвестно";
    }

    public List<string> Awards => new List<string>(_awards);
    public bool IsBlockbuster
    {
        get => _isBlockbuster;
        set => _isBlockbuster = value;
    }

    public override string AgeRating => Genre.ToLower() == "ужасы" ? "R" : "PG-13";

    public List<string> Sequels => new List<string>(_sequels);

    public void AddSequel(string sequelTitle)
    {
        _sequels.Add(sequelTitle);
        HasSequel = true;
        Console.WriteLine($"Добавлено продолжение: {sequelTitle}");
    }

    public void AddAward(string award)
    {
        if (!string.IsNullOrWhiteSpace(award) && !_awards.Contains(award))
        {
            _awards.Add(award);
            Console.WriteLine($"Фильм '{Title}' получил награду: {award}");
        }
    }

    public void AddAwards(params string[] awards)
    {
        foreach (var award in awards)
        {
            AddAward(award);
        }
    }

    public string GetAwardsInfo()
    {
        if (_awards.Count == 0)
            return "Наград нет";

        return $"Полученные награды ({_awards.Count}): {string.Join(", ", _awards)}";
    }

    public void CheckBlockbusterStatus()
    {
        _isBlockbuster = Budget > 100000000m || _awards.Count > 2;
        Console.WriteLine(_isBlockbuster ? 
            $"'{Title}' считается блокбастером!" : 
            $"'{Title}' не является блокбастером");
    }

    public override decimal CalculateProductionCost()
    {
        decimal baseCost = base.CalculateProductionCost();
        return _isBlockbuster ? baseCost * 1.3m : baseCost;
    }

    public string GetGenreDescription()
    {
        return Genre.ToLower() switch
        {
            "комедия" => "Юмор и развлечения",
            "драма" => "Эмоциональная глубина",
            "боевик" => "Экшн и приключения",
            "фэнтези" => "Магия и воображение",
            _ => "Различные темы"
        };
    }

    public override void Rate(double score)
    {
        double adjustedScore = score * GetGenreCoefficient();
        base.Rate(adjustedScore);
        Console.WriteLine($"С учетом жанра '{Genre}' (коэффициент: {GetGenreCoefficient():F2})");
    }

    private double GetGenreCoefficient()
    {
        return Genre.ToLower() switch
        {
            "комедия" => 0.9,
            "драма" => 1.1,
            "боевик" => 0.95,
            "фэнтези" => 1.05,
            _ => 1.0
        };
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $"\nТип: Игровой\nЖанр: {Genre}";
    }

    public override string GetInfo(bool includeDetails)
    {
        if (!includeDetails)
            return GetInfo();
        
        string sequelInfo = HasSequel ? $"Продолжения: {string.Join(", ", Sequels)}" : "Продолжений нет";
        return base.GetInfo(includeDetails) + $"\nЖанр: {Genre} ({GetGenreDescription()})\n{sequelInfo}";
    }

    public override decimal CalculateRevenue()
    {
        decimal baseRevenue = base.CalculateRevenue();
        decimal genreMultiplier = Genre.ToLower() switch
        {
            "боевик" => 1.5m,
            "фэнтези" => 1.3m,
            "комедия" => 1.2m,
            "драма" => 1.1m,
            _ => 1.0m
        };
        return baseRevenue * genreMultiplier;
    }

    public void PromoteFilm(string promotionType)
    {
        Console.WriteLine($"Продвижение фильма '{Title}': {promotionType}");
        if (promotionType.ToLower().Contains("акция"))
        {
            Console.WriteLine("Специальная акция: скидка 20% на билеты!");
        }
    }
}

public class AnimatedMovie : Movie
{
    private string _animationStudio;
    private string _targetAudience;
    private string _animationStyle;
    private bool _is3D;
    private int _frameRate;
    private string _renderQuality;
    private List<string> _animators;

    public AnimatedMovie(string title, string director, int releaseYear, int duration, string country, string animationStudio, string targetAudience)
        : base(title, director, releaseYear, duration, country)
    {
        AnimationStudio = animationStudio;
        TargetAudience = targetAudience;
        AnimationStyle = "2D компьютерная";
        Is3D = false;
        FrameRate = 24; 
        RenderQuality = "High";
        _animators = new List<string>();
    }

    public string AnimationStudio
    {
        get => _animationStudio;
        set => _animationStudio = !string.IsNullOrWhiteSpace(value) ? value : "Неизвестная студия";
    }

    public string TargetAudience
    {
        get => _targetAudience;
        set => _targetAudience = !string.IsNullOrWhiteSpace(value) ? value : "Для всей семьи";
    }

    public string AnimationStyle
    {
        get => _animationStyle;
        set => _animationStyle = !string.IsNullOrWhiteSpace(value) ? value : "2D компьютерная";
    }

    public bool Is3D
    {
        get => _is3D;
        set => _is3D = value;
    }
    public int FrameRate
    {
        get => _frameRate;
        set => _frameRate = value >= 12 ? value : 24;
    }

    public string RenderQuality
    {
        get => _renderQuality;
        set => _renderQuality = !string.IsNullOrWhiteSpace(value) ? value : "High";
    }

    public List<string> Animators => new List<string>(_animators);
    public override string AgeRating => TargetAudience.ToLower().Contains("дети") ? "G" : "PG";

    public void ConvertTo3D()
    {
        Is3D = true;
        AnimationStyle = "3D компьютерная";
        Console.WriteLine($"Фильм '{Title}' преобразован в 3D формат");
    }

    public bool IsSuitableForChildren()
    {
        return TargetAudience.ToLower().Contains("дети") || 
               TargetAudience.ToLower().Contains("семья") ||
               TargetAudience.ToLower().Contains("вся");
    }
    public void AddAnimator(string animator)
    {
        if (!string.IsNullOrWhiteSpace(animator) && !_animators.Contains(animator))
        {
            _animators.Add(animator);
            Console.WriteLine($"Аниматор {animator} добавлен к фильму '{Title}'");
        }
    }
    public void ChangeRenderQuality(string quality)
    {
        RenderQuality = quality;
        Console.WriteLine($"Качество рендера изменено на: {quality}");
    }
    public void OptimizeForPlatform(string platform)
    {
        FrameRate = platform.ToLower() switch
        {
            "телевидение" => 30,
            "кино" => 24,
            "веб" => 60,
            _ => 24
        };
        Console.WriteLine($"Оптимизировано для {platform}: {FrameRate} fps");
    }
    public string GetTechnicalInfo()
    {
        return $"Технические характеристики: {FrameRate} fps, {RenderQuality} quality, 3D: {Is3D}";
    }
    public override decimal CalculateProductionCost()
    {
        decimal baseCost = base.CalculateProductionCost();
        decimal animationMultiplier = Is3D ? 2.0m : 1.5m;
        return baseCost * animationMultiplier;
    }
    public override void Watch()
    {
        Console.WriteLine($"Смотрим анимацию: {Title}");
        Console.WriteLine($"Студия: {AnimationStudio}");
        Console.WriteLine($"Стиль анимации: {AnimationStyle}");
        Console.WriteLine(IsSuitableForChildren() ? "Отлично подходит для детей!" : "Рекомендуется для взрослой аудитории");
    }

    public override void AddActor(string actor)
    {
        base.AddActor(actor);
        Console.WriteLine($"(Озвучка: {actor})");
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $"\nТип: Анимационный\nСтудия: {AnimationStudio}";
    }

    public override string GetInfo(bool includeDetails)
    {
        if (!includeDetails)
            return GetInfo();
        
        return base.GetInfo(includeDetails) + $"\nСтудия: {AnimationStudio}\nАудитория: {TargetAudience}\nСтиль: {AnimationStyle}\n3D: {(Is3D ? "Да" : "Нет")}";
    }

    public override decimal CalculateRevenue()
    {
        decimal baseMultiplier = Is3D ? 3.0m : 2.5m;
        return Budget * baseMultiplier;
    }

    public void CreateMerchandise(string productType)
    {
        Console.WriteLine($"Создан мерч для '{Title}': {productType}");
        if (IsSuitableForChildren())
        {
            Console.WriteLine("Детский мерч доступен в продаже!");
        }
    }
}
Console.WriteLine("=== РАСШИРЕННАЯ СИСТЕМА УПРАВЛЕНИЯ ФИЛЬМАМИ ===\n");

        MovieCollection<Movie>.GlobalCollectionChanged += (name, change, count) => 
        {
            Console.WriteLine($" Глобальное уведомление: {name} - {change} -> {count} фильмов");
        };

        using (var collectionManager = new MovieCollectionManager())
        {
            var allMovies = new MovieCollection<Movie>();
            var featureFilms = new MovieCollection<FeatureFilm>();
            
            allMovies.MovieAdded += movie => 
                Console.WriteLine($"Локальное событие: '{movie.Title}' добавлен!");
            allMovies.MovieRemoved += movie => 
                Console.WriteLine($" Локальное событие: '{movie.Title}' удален!");

            collectionManager.RegisterCollection(allMovies);
            collectionManager.RegisterCollection(featureFilms);

            var advancedService = new AdvancedMovieService<Movie>(allMovies);
            var featureService = new AdvancedMovieService<FeatureFilm>(featureFilms);

            advancedService.OnCustomAction = (movie, action) => 
            {
                Console.WriteLine($" Кастомное действие '{action}' для фильма '{movie.Title}'");
            };

            var documentary = new Documentary("Планета Земля II", "Дэвид Аттенборо", 2016, 180, "Великобритания", "Природа и животные", true)
            {
                Producer = "BBC",
                Budget = 5000000m,
                Language = "Английский"
            };

            var featureFilm = new FeatureFilm("Начало", "Кристофер Нолан", 2010, 148, "США", "Фантастика", 160000000m)
            {
                Producer = "Уорнер Бразерс",
                ScreenplayAuthor = "Кристофер Нолан"
            };

            var animatedMovie = new AnimatedMovie("Тайна Коко", "Эдриан Молина", 2017, 105, "США", "Pixar", "Для всей семьи")
            {
                Producer = "Pixar Animation Studios",
                Budget = 175000000m
            };

            documentary.AddTags("природа", "животные", "документалистика", "наука");
            documentary.ApproveScientificReview();
            documentary.FundingSource = "государственный";

            featureFilm.AddTags("фантастика", "триллер", "сны", "экшен");
            featureFilm.AddAwards("Оскар - лучшие визуальные эффекты", "BAFTA - лучший звук");
            featureFilm.CheckBlockbusterStatus();

            animatedMovie.AddTags("анимация", "музыка", "семья", "мексика");
            animatedMovie.AddAnimator("Ли Анкрич");
            animatedMovie.OptimizeForPlatform("кино");
            animatedMovie.ConvertTo3D();

            featureFilm.AddActor("Леонардо ДиКаприо");
            featureFilm.AddActor("Эллен Пейдж", "Архитектор");
            featureFilm.AddActor("Том Харди", "Эмс");
            
            animatedMovie.AddActor("Энтони Гонсалес", "Мигель");
            documentary.AddActor("Дэвид Аттенборо", "Рассказчик");

            advancedService.AddMovieWithValidation(documentary);
            advancedService.AddMovieWithValidation(featureFilm);
            advancedService.AddMovieWithValidation(animatedMovie);

            Console.WriteLine("\n=== НОВЫЕ ВОЗМОЖНОСТИ ===");
            
            advancedService.SimulateMovieViews(featureFilm, 5);
            advancedService.SimulateMovieViews(animatedMovie, 12);
            advancedService.SimulateMovieViews(documentary, 3);

            var natureMovies = advancedService.FindMoviesByTag("природа");
            Console.WriteLine($"\nФильмы о природе: {natureMovies.Count}");

            var popularMovies = advancedService.FindPopularMovies(10);
            Console.WriteLine($"Популярные фильмы (≥10 просмотров): {popularMovies.Count}");

            advancedService.PerformCustomAction(featureFilm, "РЕКОМЕНДОВАТЬ");
            advancedService.PerformCustomAction(animatedMovie, "СКИДКА 20%");

            Console.WriteLine("\n=== РАСШИРЕННЫЙ ПОЛИМОРФИЗМ ===");
            Movie[] movies = { documentary, featureFilm, animatedMovie };

            foreach (var movie in movies)
            {
                Console.WriteLine($"\n{new string('=', 50)}");
                Console.WriteLine($"Фильм: {movie.Title}");
                Console.WriteLine($"Возрастной рейтинг: {movie.AgeRating}");
                Console.WriteLine($"Популярность: {movie.GetPopularityStatus()}");
                Console.WriteLine($"Теги: {string.Join(", ", movie.Tags)}");
                Console.WriteLine($"Затраты на производство: ${movie.CalculateProductionCost():N0}");
                
                if (movie is Documentary doc)
                {
                    Console.WriteLine($"Финансирование: {doc.GetFundingInfo()}");
                }
                else if (movie is FeatureFilm feature)
                {
                    Console.WriteLine($"Награды: {feature.GetAwardsInfo()}");
                    Console.WriteLine($"Блокбастер: {feature.IsBlockbuster}");
                }
                else if (movie is AnimatedMovie anim)
                {
                    Console.WriteLine($"Техническая информация: {anim.GetTechnicalInfo()}");
                }
            }
            Console.WriteLine("\n=== ФИЛЬТРАЦИЯ С ДЕЛЕГАТАМИ ===");
            
            var highRated = allMovies.FilterMovies(m => m.Rating >= 8.5);
            Console.WriteLine($"Фильмы с рейтингом ≥8.5: {highRated.Count}");

            var recentMovies = allMovies.FilterMovies(m => m.AddedDate > DateTime.Now.AddHours(-1));
            Console.WriteLine($"Недавно добавленные фильмы: {recentMovies.Count}");

            var nolanMovies = allMovies.FilterMovies(m => m.Director.Contains("Нолан"));
            Console.WriteLine($"Фильмы Нолана: {nolanMovies.Count}");

            Console.WriteLine("\n=== ТЕСТИРОВАНИЕ СОБЫТИЙ ===");
            allMovies.RemoveMovie("Начало");

            allMovies.DisplayAllMovies();
        }


=== РАСШИРЕННАЯ СИСТЕМА УПРАВЛЕНИЯ ФИЛЬМАМИ ===

Ресурсы коллекции загружены
Ресурсы коллекции загружены
Добавлен тег 'природа' к фильму 'Планета Земля II'
Добавлен тег 'животные' к фильму 'Планета Земля II'
Добавлен тег 'документалистика' к фильму 'Планета Земля II'
Добавлен тег 'наука' к фильму 'Планета Земля II'
Научный обзор для 'Планета Земля II' одобрен
Добавлен тег 'фантастика' к фильму 'Начало'
Добавлен тег 'триллер' к фильму 'Начало'
Добавлен тег 'сны' к фильму 'Начало'
Добавлен тег 'экшен' к фильму 'Начало'
Фильм 'Начало' получил награду: Оскар - лучшие визуальные эффекты
Фильм 'Начало' получил награду: BAFTA - лучший звук
'Начало' считается блокбастером!
Добавлен тег 'анимация' к фильму 'Тайна Коко'
Добавлен тег 'музыка' к фильму 'Тайна Коко'
Добавлен тег 'семья' к фильму 'Тайна Коко'
Добавлен тег 'мексика' к фильму 'Тайна Коко'
Аниматор Ли Анкрич добавлен к фильму 'Тайна Коко'
Оптимизировано для кино: 24 fps
Фильм 'Тайна Коко' преобразован в 3D формат
Актер Эллен Пейдж доба